# SGLang: Structured Generation & RadixAttention

This notebook explores SGLang's key innovations:
- **RadixAttention**: KV cache reuse via radix tree prefix matching
- **Constrained decoding**: JSON schema, regex, grammar-guided generation
- **Multi-step reasoning & function calling** with structured outputs
- **Comparison with vLLM** guided decoding performance

In [ ]:
import sys
sys.path.insert(0, '../../..')

import json
import time
import numpy as np
from content.utils.benchmark import BenchmarkTimer
from content.utils.latency import LatencyTracker

# Install SGLang if needed
# !pip install sglang[all] openai pydantic

## 1. RadixAttention: Prefix-Aware KV Cache Reuse

SGLang's RadixAttention stores KV caches in a radix tree indexed by token sequences.
When a new request shares a prefix with a cached sequence, the KV cache is reused —
eliminating redundant prefill computation. This is especially powerful for:
- Few-shot prompting (shared system prompt + examples)
- Multi-turn chat (prior turns cached)
- Batch requests with common prefixes

In [ ]:
import openai

# Connect to SGLang server (launch separately: python -m sglang.launch_server --model meta-llama/Llama-3.1-8B-Instruct --port 30000)
client = openai.OpenAI(base_url="http://localhost:30000/v1", api_key="none")

# Demonstrate RadixAttention benefit: repeated prefix reuse
SHARED_PREFIX = """You are an expert data scientist. Given a dataset description, provide:
1. Recommended preprocessing steps
2. Suitable model architectures
3. Evaluation metrics
Be concise and specific."""

queries = [
    "Dataset: 10M rows of clickstream data with 200 sparse categorical features.",
    "Dataset: 50K medical images (chest X-rays) with binary labels.",
    "Dataset: 1M time-series records from IoT sensors, 5-minute intervals.",
    "Dataset: 500K text reviews with star ratings (1-5).",
]

# First pass: cold cache
cold_latencies = []
for q in queries:
    t0 = time.perf_counter()
    client.chat.completions.create(
        model="default", messages=[{"role": "system", "content": SHARED_PREFIX}, {"role": "user", "content": q}],
        max_tokens=150, temperature=0
    )
    cold_latencies.append(time.perf_counter() - t0)

# Second pass: warm cache (prefix already in radix tree)
warm_latencies = []
for q in queries:
    t0 = time.perf_counter()
    client.chat.completions.create(
        model="default", messages=[{"role": "system", "content": SHARED_PREFIX}, {"role": "user", "content": q}],
        max_tokens=150, temperature=0
    )
    warm_latencies.append(time.perf_counter() - t0)

print("RadixAttention Prefix Reuse Benchmark")
print(f"{'Pass':<10} {'Mean (ms)':<12} {'P50 (ms)':<12} {'P99 (ms)':<12}")
for name, lats in [("Cold", cold_latencies), ("Warm", warm_latencies)]:
    arr = np.array(lats) * 1000
    print(f"{name:<10} {arr.mean():<12.1f} {np.median(arr):<12.1f} {np.percentile(arr, 99):<12.1f}")
print(f"\nSpeedup from prefix caching: {np.mean(cold_latencies)/np.mean(warm_latencies):.2f}x")

## 2. JSON Schema Constrained Generation

SGLang enforces output structure at the token level using finite-state machines.
Unlike post-hoc parsing, this guarantees valid JSON matching a Pydantic schema.

In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class ExtractedEntity(BaseModel):
    name: str
    entity_type: str  # person, org, location
    confidence: float
    context: Optional[str] = None

class ExtractionResult(BaseModel):
    entities: List[ExtractedEntity]
    summary: str

schema = ExtractionResult.model_json_schema()

text = """OpenAI CEO Sam Altman met with EU Commissioner Thierry Breton in Brussels 
to discuss the AI Act implementation timeline for 2025."""

timer = BenchmarkTimer("json_schema_gen")
with timer:
    resp = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": f"Extract entities from: {text}"}],
        response_format={"type": "json_schema", "json_schema": {"name": "extraction", "schema": schema}},
        max_tokens=300, temperature=0
    )

result = json.loads(resp.choices[0].message.content)
parsed = ExtractionResult(**result)
print(f"Extracted {len(parsed.entities)} entities in {timer.elapsed_ms:.1f}ms:")
for e in parsed.entities:
    print(f"  [{e.entity_type}] {e.name} (conf={e.confidence:.2f})")
print(f"Summary: {parsed.summary}")

## 3. Regex-Constrained Generation

For cases where JSON schema is too rigid, SGLang supports regex constraints.
The FSM masks invalid tokens at each step, ensuring output matches the pattern.

In [ ]:
# Regex patterns for structured extraction
patterns = {
    "ip_address": r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}",
    "date_iso": r"\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])",
    "semver": r"v\d+\.\d+\.\d+(?:-[a-z]+\.\d+)?",
    "email": r"[a-z][a-z0-9.]+@[a-z]+\.[a-z]{2,4}",
}

prompts = {
    "ip_address": "What is a common DNS server IP address?",
    "date_iso": "When was Python 3.12 released? Give the date.",
    "semver": "What is the latest stable version of Node.js?",
    "email": "Generate a sample contact email for a tech company.",
}

print(f"{'Pattern':<12} {'Output':<30} {'Latency (ms)':<12}")
print("-" * 54)
for name, regex in patterns.items():
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": prompts[name]}],
        extra_body={"regex": regex},
        max_tokens=50, temperature=0
    )
    lat = (time.perf_counter() - t0) * 1000
    output = resp.choices[0].message.content.strip()
    print(f"{name:<12} {output:<30} {lat:<12.1f}")

## 4. Multi-Step Reasoning with Structured Outputs

SGLang's frontend DSL enables chaining multiple generation calls with
constrained outputs at each step — ideal for chain-of-thought with verified structure.

In [ ]:
class ReasoningStep(BaseModel):
    thought: str
    action: str  # "calculate", "lookup", "conclude"
    result: str

class ReasoningChain(BaseModel):
    steps: List[ReasoningStep]
    final_answer: str

problem = """A data center has 8 racks, each with 4 servers. Each server has 8 GPUs 
with 80GB HBM each. If KV cache uses 2 bytes per element, sequence length is 8192, 
num_heads=32, head_dim=128, and num_layers=80, what fraction of total GPU memory 
is consumed by KV cache when all GPUs serve one model with tensor parallelism?"""

chain_schema = ReasoningChain.model_json_schema()

t0 = time.perf_counter()
resp = client.chat.completions.create(
    model="default",
    messages=[{"role": "user", "content": f"Solve step by step: {problem}"}],
    response_format={"type": "json_schema", "json_schema": {"name": "reasoning", "schema": chain_schema}},
    max_tokens=800, temperature=0
)
chain_lat = (time.perf_counter() - t0) * 1000

chain = ReasoningChain(**json.loads(resp.choices[0].message.content))
print(f"Multi-step reasoning ({len(chain.steps)} steps, {chain_lat:.0f}ms):")
for i, step in enumerate(chain.steps, 1):
    print(f"  Step {i} [{step.action}]: {step.thought[:80]}...")
    print(f"    -> {step.result}")
print(f"\nFinal: {chain.final_answer}")

## 5. Function Calling with SGLang

SGLang supports OpenAI-compatible function calling with constrained argument generation,
ensuring tool arguments always parse correctly.

In [ ]:
tools = [
    {"type": "function", "function": {
        "name": "get_gpu_metrics",
        "description": "Get real-time GPU utilization and memory stats",
        "parameters": {"type": "object", "properties": {
            "gpu_id": {"type": "integer", "description": "GPU device index"},
            "metrics": {"type": "array", "items": {"type": "string", "enum": ["utilization", "memory", "temperature", "power"]}}
        }, "required": ["gpu_id", "metrics"]}
    }},
    {"type": "function", "function": {
        "name": "scale_replicas",
        "description": "Scale model serving replicas up or down",
        "parameters": {"type": "object", "properties": {
            "model_name": {"type": "string"},
            "replicas": {"type": "integer", "minimum": 1, "maximum": 32},
            "reason": {"type": "string"}
        }, "required": ["model_name", "replicas"]}
    }}
]

queries_fc = [
    "Check GPU 0 memory and temperature.",
    "Scale llama-70b to 4 replicas because latency is too high.",
    "What's the power draw on GPU 3?",
]

print("Function Calling Results:")
print("=" * 60)
for q in queries_fc:
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": q}],
        tools=tools, tool_choice="auto", max_tokens=200, temperature=0
    )
    lat = (time.perf_counter() - t0) * 1000
    tc = resp.choices[0].message.tool_calls[0] if resp.choices[0].message.tool_calls else None
    if tc:
        args = json.loads(tc.function.arguments)
        print(f"  Q: {q}")
        print(f"  -> {tc.function.name}({args}) [{lat:.0f}ms]")
    print()

## 6. SGLang vs vLLM: Guided Decoding Comparison

Both engines support constrained generation, but differ in implementation:
- **SGLang**: Compressed FSM with RadixAttention, jump-forward optimization
- **vLLM**: Outlines-based guided decoding with xgrammar backend

We benchmark both on identical structured generation tasks.

In [ ]:
# Assumes vLLM running on port 8000, SGLang on port 30000
sglang_client = openai.OpenAI(base_url="http://localhost:30000/v1", api_key="none")
vllm_client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="none")

class BenchmarkOutput(BaseModel):
    category: str
    score: float
    tags: List[str]
    explanation: str

bench_schema = BenchmarkOutput.model_json_schema()
bench_prompt = "Classify this text and rate quality: 'SGLang achieves 5x throughput over vLLM on structured generation tasks.'"

N_RUNS = 10
results = {"sglang": [], "vllm": []}

for engine, cl in [("sglang", sglang_client), ("vllm", vllm_client)]:
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        try:
            resp = cl.chat.completions.create(
                model="default",
                messages=[{"role": "user", "content": bench_prompt}],
                response_format={"type": "json_schema", "json_schema": {"name": "bench", "schema": bench_schema}},
                max_tokens=200, temperature=0
            )
            results[engine].append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f"  {engine} error: {e}")
            break

print("\nStructured Generation Latency Comparison (ms)")
print(f"{'Engine':<10} {'Mean':<10} {'P50':<10} {'P95':<10} {'P99':<10}")
print("-" * 50)
for engine, lats in results.items():
    if lats:
        arr = np.array(lats)
        print(f"{engine:<10} {arr.mean():<10.1f} {np.median(arr):<10.1f} {np.percentile(arr, 95):<10.1f} {np.percentile(arr, 99):<10.1f}")

if results["sglang"] and results["vllm"]:
    speedup = np.mean(results["vllm"]) / np.mean(results["sglang"])
    print(f"\nSGLang speedup over vLLM: {speedup:.2f}x")
    print(f"Key advantage: RadixAttention reuses prefix KV cache across structured generation calls")